In [1]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    raw_data_path:Path


In [2]:
%pwd

'c:\\Customer_churn_project\\notebooks'

In [3]:
import os
os.chdir("../")

In [4]:
from src.CustomerChurnPrediction.constants import *
from src.CustomerChurnPrediction.utils.common import read_yaml, create_directories

In [5]:
@dataclass(frozen=True)
class DatabaseConfig:
    database_name:str
    table_name:str

In [6]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    raw_data_path:Path
    database_info: DatabaseConfig

In [7]:
class ConfigurationManager:
    def __init__(self,config_filepath = CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_ingetion_config(self):
        config = self.config.data_ingestion
        db = config.database

        create_directories([config.root_dir])

        database_config = DatabaseConfig(
            database_name=db.database_name,
            table_name=db.table_name
        )

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            raw_data_path=config.raw_data_path,
            database_info=database_config
        )

        return data_ingestion_config

In [8]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from src.CustomerChurnPrediction.utils.logger import logger
from src.CustomerChurnPrediction.utils.exception import CustomException

load_dotenv()

True

In [9]:
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config = config

            
    
    def read_sql_data(self,database_name: str, table_name: str) -> pd.DataFrame:
        
        try:
            logger.info(f"Connecting to database: {database_name}")

            # read credentials from .env
            user     = os.getenv("DB_USER")
            password = os.getenv("DB_PASSWORD")
            host     = os.getenv("DB_HOST")
            port     = os.getenv("DB_PORT")

            # create connection
            connection_url = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database_name}"
            engine         = create_engine(connection_url)

            # fetch data
            df = pd.read_sql(f"SELECT * FROM {table_nam}", engine)

            logger.info(f"Data fetched successfully: {df.shape[0]} rows, {df.shape[1]} columns")

            return df
        except Exception as e:
            raise CustomException(e,sys)

    def save_raw_data(self):

        database_name = self.config.database_info.database_name
        table_name = self.config.database_info.table_name
        df = self.read_sql_data(database_name,table_name)
        create_directories([os.path.dirname(self.config.raw_data_path)])
        df.to_csv(self.config.raw_data_path)

In [11]:
try:
    config = ConfigurationManager()
    data_ingetion_config = config.get_data_ingetion_config()
    data_ingetion = DataIngestion(data_ingetion_config)
    data_ingetion.save_raw_data()
except Exception as e:
    raise CustomException(e,sys)

[2026-06-21 11:19:10,158] 36 CustomerChurnPrediction - INFO - yaml file: config\config.yml loaded successfully
[2026-06-21 11:19:10,160] 53 CustomerChurnPrediction - INFO - created directory at: artifacts
[2026-06-21 11:19:10,161] 53 CustomerChurnPrediction - INFO - created directory at: artifacts/data_ingestion
[2026-06-21 11:19:10,162] 10 CustomerChurnPrediction - INFO - Connecting to database: churn_db


CustomException: 
==================================================
File: C:\Users\Admin\AppData\Local\Temp\ipykernel_12316\3582102100.py
Line: 5
Error: 
==================================================
File: C:\Users\Admin\AppData\Local\Temp\ipykernel_12316\2143313473.py
Line: 23
Error: name 'table_nam' is not defined
==================================================
==================================================